In [23]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Paths ────────────────────────────────────────────────────────────────
TPDNE_DIR = '/kaggle/input/datasets/dhathrikarthik/website-gan-fake/data/tpdne_fakes'
WEIGHTS_PATH  = '/kaggle/input/datasets/dhathrikarthik/deepfake-classifier-weights/deepfake_classifier.pth'
FACES140K_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces'

# Load existing model
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

state = torch.load(WEIGHTS_PATH, map_location=DEVICE)
state = {k.replace('model.', '', 1): v for k, v in state.items()}  # strip prefix — required for this checkpoint
model.load_state_dict(state)

model.to(DEVICE)
print("✅ Loaded existing weights")

Using device: cuda
✅ Loaded existing weights


In [24]:
# Find the actual real-images folder inside the 140k dataset
# (commonly something like real_vs_fake/train/real or similar — check with !ls)
print(os.listdir(FACES140K_DIR))  # run this first to confirm structure

['valid.csv', 'real_vs_fake', 'train.csv', 'test.csv']


In [25]:
# Once you know the real folder path, set it here:
REAL_DIR = f'{FACES140K_DIR}/real_vs_fake/real-vs-fake/train/real'  # ADJUST based on actual structure

all_real_files = [os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.seed(42)
real_subset = random.sample(all_real_files, min(600, len(all_real_files)))

tpdne_files = [os.path.join(TPDNE_DIR, f) for f in os.listdir(TPDNE_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Real images: {len(real_subset)}")
print(f"TPDNE fakes: {len(tpdne_files)}")

# Build (path, label) pairs — 0 = real, 1 = fake
data = [(p, 0) for p in real_subset] + [(p, 1) for p in tpdne_files]
random.shuffle(data)

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42, 
                                          stratify=[label for _, label in data])
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

Real images: 600
TPDNE fakes: 300
Train: 720, Val: 180


In [12]:
print(os.listdir('/kaggle/input/datasets/dhathrikarthik'))
print(os.listdir('/kaggle/input/datasets/xhlulu'))

['deepfake-classifier-weights', 'website-gan-fake']
['140k-real-and-fake-faces']


In [26]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FineTuneDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = FineTuneDataset(train_data, transform=transform)
val_dataset   = FineTuneDataset(val_data, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [27]:
criterion = nn.CrossEntropyLoss()
# Low LR — we're patching, not relearning. 10-20x smaller than your original training LR.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss/train_total:.4f} | "
          f"Train Acc: {train_correct/train_total*100:.2f}% | "
          f"Val Acc: {val_correct/val_total*100:.2f}%")

Epoch 1/5 | Train Loss: 1.8086 | Train Acc: 72.22% | Val Acc: 78.89%
Epoch 2/5 | Train Loss: 1.4950 | Train Acc: 72.92% | Val Acc: 78.89%
Epoch 3/5 | Train Loss: 1.4323 | Train Acc: 75.97% | Val Acc: 80.00%
Epoch 4/5 | Train Loss: 1.2730 | Train Acc: 76.81% | Val Acc: 80.00%
Epoch 5/5 | Train Loss: 1.2376 | Train Acc: 76.94% | Val Acc: 81.11%


In [28]:
torch.save(model.state_dict(), 'deepfake_classifier_finetuned.pth')
print("✅ Saved fine-tuned weights as deepfake_classifier_finetuned.pth")

✅ Saved fine-tuned weights as deepfake_classifier_finetuned.pth


In [22]:
print(os.listdir('/kaggle/input/datasets/dhathrikarthik/website-gan-fake/data'))

['tpdne_fakes']
